# Titanic Feature Engineering Pipeline - Max Version

EDA, Feature Engineering, Feature Selection, Pipeline, GridSearchCV, SHAP, AutoML-style 비교를 포함한 과제용 노트북입니다. GitHub 저장소에는 `titanic.csv`와 이 노트북을 함께 업로드하면 됩니다.

In [ ]:
import os, warnings, json
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance
try:
    import shap
except Exception:
    shap = None
OUT = './titanic_outputs'
os.makedirs(OUT, exist_ok=True)

## STEP 01. 데이터 준비

In [ ]:
df = pd.read_csv('titanic.csv')
print('Shape:', df.shape)
display(df.head())
display(df.info())
TARGET = 'Survived'
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## STEP 02. EDA

In [ ]:
missing = df.isna().mean().mul(100).round(2).sort_values(ascending=False)
display(missing)
fig, axes = plt.subplots(2,2,figsize=(11,8))
df['Survived'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0]); axes[0,0].set_title('Target Distribution')
axes[0,1].hist(df['Age'].dropna(), bins=30); axes[0,1].set_title('Age Distribution')
df.boxplot(column='Fare', by='Survived', ax=axes[1,0]); axes[1,0].set_title('Fare Outliers by Survival')
df.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[1,1]); axes[1,1].set_title('Survival Rate by Sex')
plt.suptitle(''); plt.tight_layout(); plt.show()
plt.figure(figsize=(7,5)); sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, fmt='.2f', cmap='Blues'); plt.title('Correlation Heatmap'); plt.show()

## STEP 03. 특성 공학 파이프라인 구현

In [ ]:
class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X['FamilySize'] = X['SibSp'].fillna(0) + X['Parch'].fillna(0) + 1
        X['IsAlone'] = (X['FamilySize'] == 1).astype(int)
        X['FarePerPerson'] = X['Fare'] / X['FamilySize'].replace(0, 1)
        X['HasCabin'] = X['Cabin'].notna().astype(int)
        X['Title'] = X['Name'].str.extract(r',\s*([^\.]+)\.', expand=False).fillna('Unknown')
        X['Title'] = X['Title'].where(X['Title'].isin(['Mr','Miss','Mrs','Master']), 'Rare')
        X['AgeGroup'] = pd.cut(X['Age'], bins=[-1,12,18,35,60,120], labels=['Child','Teen','YoungAdult','Adult','Senior']).astype(object)
        X['AgeGroup'] = X['AgeGroup'].fillna('Missing')
        return X

NUMERIC = ['Pclass','Age','SibSp','Parch','Fare','FamilySize','IsAlone','FarePerPerson','HasCabin']
CATEGORICAL = ['Sex','Embarked','Title','AgeGroup']

def make_preprocessor(num_strategy='median', encoding='onehot', scaler='standard', select_k=None):
    scaler_obj = {'standard':StandardScaler(), 'minmax':MinMaxScaler(), 'robust':RobustScaler(), 'none':'passthrough'}[scaler]
    encoder_obj = OneHotEncoder(handle_unknown='ignore', sparse_output=False) if encoding=='onehot' else OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    num_pipe = Pipeline([('imputer', SimpleImputer(strategy=num_strategy)), ('scaler', scaler_obj)])
    cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', encoder_obj)])
    pre = ColumnTransformer([('num', num_pipe, NUMERIC), ('cat', cat_pipe, CATEGORICAL)], remainder='drop', verbose_feature_names_out=False)
    steps = [('feature_engineering', TitanicFeatureEngineer()), ('preprocess', pre)]
    if select_k is not None:
        steps.append(('feature_selection', SelectKBest(mutual_info_classif, k=select_k)))
    return Pipeline(steps)

## STEP 04-05. 실험 비교 및 모델 평가

In [ ]:
def evaluate_pipeline(name, preprocess_pipe, model):
    clf = Pipeline([('prep', preprocess_pipe), ('model', model)])
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    prob = clf.predict_proba(X_test)[:,1]
    return {'Experiment':name, 'Model':model.__class__.__name__, 'Accuracy':accuracy_score(y_test,pred), 'Precision':precision_score(y_test,pred), 'Recall':recall_score(y_test,pred), 'F1':f1_score(y_test,pred), 'ROC_AUC':roc_auc_score(y_test,prob)}, clf

configs = [
    ('Base','median','ordinal','none',None),
    ('Exp-1','mean','onehot','standard',None),
    ('Exp-2','median','ordinal','minmax',10),
    ('Exp-3','most_frequent','onehot','robust',12),
]
models = [LogisticRegression(max_iter=1000, random_state=42), RandomForestClassifier(n_estimators=120, random_state=42, class_weight='balanced')]
rows=[]; fitted={}
for exp, num_s, enc, scale, k in configs:
    for m in models:
        r, p = evaluate_pipeline(exp, make_preprocessor(num_s, enc, scale, k), m)
        rows.append(r); fitted[(exp, r['Model'])]=p
results = pd.DataFrame(rows).sort_values('ROC_AUC', ascending=False)
display(results)
results.to_csv(os.path.join(OUT,'performance_results_max.csv'), index=False)

## 추가점수 1. Pipeline + GridSearchCV

In [ ]:
rf_pipe = Pipeline([('prep', make_preprocessor('median','onehot','robust',12)), ('model', RandomForestClassifier(random_state=42, class_weight='balanced'))])
param_grid = {'model__n_estimators':[80,120], 'model__max_depth':[3,5], 'model__min_samples_split':[2]}
grid = GridSearchCV(rf_pipe, param_grid, scoring='roc_auc', cv=StratifiedKFold(5, shuffle=True, random_state=42), n_jobs=1)
grid.fit(X_train, y_train)
print('Best Params:', grid.best_params_)
print('CV ROC-AUC:', grid.best_score_)
pred=grid.predict(X_test); prob=grid.predict_proba(X_test)[:,1]
print('Test Accuracy:', accuracy_score(y_test,pred))
print('Test F1:', f1_score(y_test,pred))
print('Test ROC-AUC:', roc_auc_score(y_test,prob))

## 추가점수 2. AutoML-style 모델 자동 비교

In [ ]:
model_zoo = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=120, max_depth=5, random_state=42, class_weight='balanced'),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=120, max_depth=5, random_state=42, class_weight='balanced'),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=60, max_depth=2, random_state=42),
}
auto_rows=[]
for name, model in model_zoo.items():
    pipe = Pipeline([('prep', make_preprocessor('median','onehot','robust',12)), ('model', model)])
    pipe.fit(X_train, y_train)
    pred=pipe.predict(X_test); prob=pipe.predict_proba(X_test)[:,1]
    auto_rows.append({'Model':name,'Accuracy':accuracy_score(y_test,pred),'F1':f1_score(y_test,pred),'ROC_AUC':roc_auc_score(y_test,prob)})
auto = pd.DataFrame(auto_rows).sort_values('ROC_AUC', ascending=False)
display(auto)
auto.to_csv(os.path.join(OUT,'automl_style_comparison.csv'), index=False)

## 추가점수 3. Feature Importance + SHAP 설명 가능성

In [ ]:
best_model = grid.best_estimator_
prep = best_model.named_steps['prep']
feature_names_pre = prep.named_steps['preprocess'].get_feature_names_out()
mask = prep.named_steps['feature_selection'].get_support()
feature_names = np.array(feature_names_pre)[mask]
imp = pd.DataFrame({'feature':feature_names,'importance':best_model.named_steps['model'].feature_importances_}).sort_values('importance', ascending=False)
display(imp.head(12))
plt.figure(figsize=(7,4)); p=imp.head(12).iloc[::-1]; plt.barh(p['feature'],p['importance']); plt.title('Feature Importance'); plt.tight_layout(); plt.show()

perm = permutation_importance(best_model, X_test, y_test, scoring='roc_auc', n_repeats=3, random_state=42, n_jobs=1)
perm_df = pd.DataFrame({'feature':X_test.columns, 'importance_mean':perm.importances_mean}).sort_values('importance_mean', ascending=False)
display(perm_df)

if shap is not None:
    X_trans = prep.transform(X_test)
    X_shap = pd.DataFrame(X_trans, columns=feature_names)
    sample = X_shap.sample(n=min(80,len(X_shap)), random_state=42)
    explainer = shap.TreeExplainer(best_model.named_steps['model'])
    shap_values = explainer.shap_values(sample)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    if getattr(sv, 'ndim', 2) == 3:
        sv = sv[:,:,1]
    shap.summary_plot(sv, sample, max_display=12)
    shap.summary_plot(sv, sample, plot_type='bar', max_display=12)
else:
    print('SHAP is not installed. Run: pip install shap')